# ⚡ ISIRI 2.0 — Fast ByT5-Small Fine-Tuning (Under 8 Minutes on T4 GPU)

Fine-tunes **`google/byt5-small`** for bidirectional Tulu ↔ English translation.
Hyper-optimized with batch size 16, sequence length 96, and Adafactor learning rate for ultra-fast GPU training.

## 1. Install Dependencies & Check GPU

In [ ]:
!pip install -q transformers datasets evaluate sacrebleu sentencepiece accelerate

import torch
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"🚀 GPU Active:   {torch.cuda.get_device_name(0)}")
else:
    print("⚠️ Switch to GPU: Runtime -> Change runtime type -> Select T4 GPU")

## 2. Load Dataset from GitHub directly

In [ ]:
import os
import pandas as pd
from datasets import Dataset, DatasetDict

# Directly fetch the latest 2,825-pair dataset from GitHub
DATASET_URL = "https://raw.githubusercontent.com/shravya-sk/ISIRI-2.0/main/datasets/processed/clean_dataset.csv"
df = pd.read_csv(DATASET_URL)
df = df.dropna(subset=["English", "Tulu"])
df["English"] = df["English"].astype(str).str.strip()
df["Tulu"] = df["Tulu"].astype(str).str.strip()
df = df[(df["English"] != "") & (df["Tulu"] != "")]

print(f"Loaded {len(df)} parallel pairs.")

# Build Bidirectional Dataset
inputs, targets, directions = [], [], []
for _, row in df.iterrows():
    en, tu = row["English"], row["Tulu"]
    # Tulu -> English
    inputs.append(f"translate Tulu to English: {tu}")
    targets.append(en)
    directions.append("tulu_to_en")
    # English -> Tulu
    inputs.append(f"translate English to Tulu: {en}")
    targets.append(tu)
    directions.append("en_to_tulu")

raw_dataset = Dataset.from_dict({
    "input_text": inputs,
    "target_text": targets,
    "direction": directions
})

train_test = raw_dataset.train_test_split(test_size=0.1, seed=42)
dataset_dict = DatasetDict({
    "train": train_test["train"],
    "test": train_test["test"]
})

print(f"Dataset: Train={len(dataset_dict['train'])}, Test={len(dataset_dict['test'])}")

## 3. Fast Tokenization

In [ ]:
from transformers import AutoTokenizer

MODEL_NAME = "google/byt5-small"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

MAX_LEN = 96

def preprocess_function(examples):
    model_inputs = tokenizer(
        examples["input_text"],
        max_length=MAX_LEN,
        truncation=True,
        padding=False
    )
    labels = tokenizer(
        text_target=examples["target_text"],
        max_length=MAX_LEN,
        truncation=True,
        padding=False
    )
    model_inputs["labels"] = labels["input_ids"]
    return model_inputs

tokenized_datasets = dataset_dict.map(
    preprocess_function,
    batched=True,
    remove_columns=["input_text", "target_text", "direction"]
)
print("Tokenization complete!")

## 4. Setup Model & Fast Trainer

In [ ]:
import torch
from transformers import (
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

model = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

data_collator = DataCollatorForSeq2Seq(
    tokenizer,
    model=model,
    label_pad_token_id=-100,
    pad_to_multiple_of=8 if torch.cuda.is_available() else None
)

training_args = Seq2SeqTrainingArguments(
    output_dir="./byt5_checkpoints",
    eval_strategy="no",
    save_strategy="no",
    learning_rate=1.5e-3,
    per_device_train_batch_size=16,
    gradient_accumulation_steps=1,
    weight_decay=0.01,
    num_train_epochs=8,
    warmup_ratio=0.05,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    logging_steps=50,
    report_to="none"
)

trainer = Seq2SeqTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    processing_class=tokenizer,
    data_collator=data_collator
)

## 5. Train Model (~6 Minutes on T4 GPU)

In [ ]:
print("Starting fast GPU training (8 epochs)...\n")
trainer.train()
print("\n🎉 Training completed successfully!")

## 6. Evaluate on Test Set

In [ ]:
import evaluate
sacrebleu = evaluate.load("sacrebleu")
chrf = evaluate.load("chrf")

def decode_bytes(seqs):
    res = []
    for seq in seqs:
        raw = bytearray([int(t) - 3 for t in seq if 3 <= int(t) <= 258])
        res.append(raw.decode("utf-8", errors="ignore").strip())
    return res

test_preds = trainer.predict(test_dataset=tokenized_datasets["test"])
decoded_preds = decode_bytes(test_preds.predictions)
decoded_labels = [[l] for l in decode_bytes(test_preds.label_ids)]

bleu_score = sacrebleu.compute(predictions=decoded_preds, references=decoded_labels)["score"]
chrf_score = chrf.compute(predictions=decoded_preds, references=decoded_labels)["score"]

print("\n=== TEST SET METRICS ===")
print(f"Test SacreBLEU: {bleu_score:.2f}")
print(f"Test chrF++:   {chrf_score:.2f}")

## 7. Interactive Translation Playground (Test Output)

In [ ]:
def translate(text, direction="tulu_to_en"):
    prefix = "translate Tulu to English: " if direction == "tulu_to_en" else "translate English to Tulu: "
    prompt = prefix + text
    inputs = tokenizer(prompt, return_tensors="pt", max_length=96, truncation=True).to(model.device)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=48,
            num_beams=2,
            early_stopping=True,
            decoder_start_token_id=0,
            eos_token_id=1,
            pad_token_id=0
        )
    return tokenizer.decode(outputs[0], skip_special_tokens=True).strip()

test_samples = [
    ("Open YouTube", "en_to_tulu"),
    ("Turn on the bedroom light", "en_to_tulu"),
    ("Turn off the fan", "en_to_tulu"),
    ("How are you?", "en_to_tulu"),
    ("youtube open malpule", "tulu_to_en"),
    ("kone da light on malpule", "tulu_to_en"),
    ("ini mangalore da weather encha undu", "tulu_to_en"),
    ("yaan illag povond ulle", "tulu_to_en"),
]

print("\n=== SAMPLE INFERENCE VERIFICATION ===")
for sentence, direction in test_samples:
    result = translate(sentence, direction)
    print(f"[{direction}] '{sentence}' -> '{result}'")

## 8. Export Model for ISIRI 2.0

In [ ]:
import os
from google.colab import files

FINAL_EXPORT = "./byt5_tulu_english"
os.makedirs(FINAL_EXPORT, exist_ok=True)

# Direct save of fine-tuned model weights
model.save_pretrained(FINAL_EXPORT)
tokenizer.save_pretrained(FINAL_EXPORT)

# Zip the model for download
!zip -r byt5_tulu_english.zip ./byt5_tulu_english

print("\n✅ Trained model saved directly and zipped as 'byt5_tulu_english.zip'!")

# Trigger browser download
try:
    files.download('byt5_tulu_english.zip')
except Exception as e:
    print("Download trigger notice:", e)